## Piping a Prompt, Model, and an Output Parser

In [1]:
pip show langchain

Name: langchain
Version: 1.2.10
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /opt/conda/lib/python3.11/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: gpt-researcher, langchain-azure-ai, langchain-tavily
Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext dotenv
%dotenv

In [3]:
from langchain_openai.chat_models import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

In [4]:
chat = ChatOpenAI(model = 'gpt-4o-mini', 
                  seed = 365, 
                  temperature = 0, 
                  max_tokens = 100)

In [5]:
list_instructions = CommaSeparatedListOutputParser().get_format_instructions()

In [6]:
list_instructions

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

In [7]:
chat_template1 = ChatPromptTemplate.from_messages([
    ('human', "I've recently adopted a {pet} which is a {breed}. Could you suggest several training tips?")])

In [8]:
chat_template = ChatPromptTemplate.from_messages([
    ('human', "I've recently adopted a {pet}. Could you suggest three {pet} names? \n" + list_instructions)])

In [9]:
print(chat_template.messages[0].prompt.template)

I've recently adopted a {pet}. Could you suggest three {pet} names? 
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [10]:
list_output_parser = CommaSeparatedListOutputParser()

In [11]:
chat_template_result = chat_template.invoke({'pet':'dog'})

In [12]:
chat_template_result

ChatPromptValue(messages=[HumanMessage(content="I've recently adopted a dog. Could you suggest three dog names? \nYour response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`", additional_kwargs={}, response_metadata={})])

In [13]:
chat_result = chat.invoke(chat_template_result)

In [14]:
chat_result

AIMessage(content='Buddy, Bella, Max', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 49, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0c3ab1c9be', 'id': 'chatcmpl-DHT227Hwl1iHEbMoDRQvwvwxnufgk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--8ae9499b-73a1-4217-91c8-9b7e97f72d11-0', usage_metadata={'input_tokens': 49, 'output_tokens': 5, 'total_tokens': 54, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
list_output_parser.invoke(chat_result)

['Buddy', 'Bella', 'Max']

In [16]:
chain = chat_template | chat | list_output_parser

In [17]:
chain.invoke({'pet':'dog'})

['Buddy', 'Bella', 'Max']

## Batching

In [20]:
chain = chat_template1 | chat

In [21]:
chain.invoke({'pet':'dog', 'breed':'shepherd'})

AIMessage(content='Congratulations on your new shepherd! Shepherds are intelligent and eager to please, which makes them great candidates for training. Here are several training tips to help you get started:\n\n1. **Start with Basic Commands**: Teach essential commands like "sit," "stay," "come," "down," and "leave it." Use positive reinforcement, such as treats or praise, to encourage good behavior.\n\n2. **Consistency is Key**: Use the same commands and cues consistently. This helps your dog understand', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 24, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0c3ab1c9be', 'id': 'chatcmpl-DHT2ueHrZWBeASFfF7e

In [22]:
%%time
chain.batch([{'pet':'dog', 'breed':'shepherd'}, 
             {'pet':'dragon', 'breed':'night fury'}])

CPU times: user 13.1 ms, sys: 15.7 ms, total: 28.8 ms
Wall time: 3.01 s


[AIMessage(content='Congratulations on your new shepherd! Shepherds are intelligent and eager to please, making them great candidates for training. Here are several training tips to help you get started:\n\n1. **Start with Basic Commands**: Teach essential commands like "sit," "stay," "come," and "down." Use positive reinforcement, such as treats and praise, to encourage good behavior.\n\n2. **Consistency is Key**: Use the same commands and gestures consistently. This helps your dog understand what you expect from them', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 24, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0dd0ffdd96', 'id': 'chatcmpl-DHT2

In [23]:
%%time
chain.invoke({'pet':'dog', 'breed':'shepherd'})

CPU times: user 16.1 ms, sys: 0 ns, total: 16.1 ms
Wall time: 1.4 s


AIMessage(content='Congratulations on your new shepherd! Shepherds are intelligent and eager to please, making them great candidates for training. Here are several training tips to help you get started:\n\n1. **Start with Basic Commands**: Teach essential commands like "sit," "stay," "come," and "down." Use positive reinforcement, such as treats and praise, to encourage good behavior.\n\n2. **Consistency is Key**: Use the same commands and gestures consistently. This helps your dog understand what you expect from them', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 24, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0dd0ffdd96', 'id': 'chatcmpl-DHT3D

In [24]:
%%time
chain.invoke({'pet':'dragon', 'breed':'night fury'})

CPU times: user 13 ms, sys: 0 ns, total: 13 ms
Wall time: 2.11 s


AIMessage(content='Training a Night Fury, like Toothless from "How to Train Your Dragon," can be a fun and rewarding experience! Here are several tips to help you train your new dragon:\n\n1. **Build Trust**: Establish a bond with your Night Fury by spending time together. Offer treats (like fish or other favorite foods) and engage in gentle play to build trust.\n\n2. **Positive Reinforcement**: Use positive reinforcement techniques. Reward your dragon with treats or praise when it follows commands or exhibits', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 25, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0dd0ffdd96', 'id': 'chatcmpl-DHT3FURS7hIaM

In [44]:
inputs = [{'pet':'dog', 'breed':'shepherd'}, 
             {'pet':'dragon', 'breed':'night fury'}]

In [45]:
%%time
chain.batch(inputs, config={"max_concurrency": 5})

CPU times: user 28.6 ms, sys: 15.9 ms, total: 44.5 ms
Wall time: 3.87 s


[AIMessage(content='Congratulations on your new shepherd! Shepherds are intelligent and eager to please, making them great candidates for training. Here are several training tips to help you get started:\n\n1. **Start with Basic Commands**: Teach essential commands like "sit," "stay," "come," "down," and "leave it." Use positive reinforcement, such as treats or praise, to encourage good behavior.\n\n2. **Consistency is Key**: Use the same commands and gestures consistently. This helps your dog understand what', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 24, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0c3ab1c9be', 'id': 'chatcmpl-DHTRF62ktKWFWq

In [46]:
response = chain.stream({'pet':'dragon', 'breed':'night fury'})

In [47]:
next(response)

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--95a1cafb-0bab-402c-96a3-f453c0ab06e0')

In [48]:
full_message = ""

while True:
    try:
        # Bir sonraki parçayı manuel olarak istiyoruz
        chunk = next(response)
        
        # Parçadaki metni ekliyoruz
        full_message += chunk.content
        print(chunk.content, end="", flush=True)
        
    except StopIteration:
        # Akış bittiğinde Python bu hatayı fırlatır, döngüden çıkıyoruz
        break

print("\n\nMesajın tamamı alındı.")

Training a Night Fury, like Toothless from "How to Train Your Dragon," can be a fun and rewarding experience! Here are several tips to help you train your new dragon:

1. **Build Trust**: Establish a bond with your Night Fury by spending time together. Offer treats (like fish or other favorite foods) and engage in gentle play to build trust.

2. **Positive Reinforcement**: Use positive reinforcement techniques. Reward your dragon with treats, praise, or affection when it follows

Mesajın tamamı alındı.


In [49]:
for i in response:
    print(i.content, end = '')

In [50]:
type(chat_template)

langchain_core.prompts.chat.ChatPromptTemplate

## Piping Chains and the RunnablePassthrough Class

In [51]:
from langchain_core.runnables import RunnablePassthrough

In [52]:
RunnablePassthrough().invoke([1, 2, 3])

[1, 2, 3]

In [53]:
chat_template_tools = ChatPromptTemplate.from_template('''
What are the five most important tools a {job title} needs?
Answer only by listing the tools.
''')

chat_template_strategy = ChatPromptTemplate.from_template('''
Considering the tools provided, develop a strategy for effectively learning and mastering them:
{tools}
''')

In [54]:
chat_template_tools

ChatPromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, template='\nWhat are the five most important tools a {job title} needs?\nAnswer only by listing the tools.\n'), additional_kwargs={})])

In [55]:
chat = ChatOpenAI(model_name = 'gpt-4', 
                  seed = 365,
                  temperature = 0,
                  max_tokens = 500)

In [56]:
from langchain_core.output_parsers import StrOutputParser

string_parser = StrOutputParser()

In [57]:
chain_tools = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()})
chain_strategy = chat_template_strategy | chat | string_parser

In [58]:
print(chain_tools.invoke({'job title':'data scientist'}))

{'tools': '1. Python\n2. R Programming\n3. SQL\n4. Tableau\n5. Hadoop'}


In [95]:
chain_tools1 = (chat_template_tools | chat)

In [96]:
print(chain_tools1.invoke({'job title':'data scientist'}))

content='1. Python\n2. R Programming\n3. SQL\n4. Tableau\n5. Hadoop' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 27, 'total_tokens': 49, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'id': 'chatcmpl-DFwVI0MNvLIPfX0yr4XGBSTe5bzmx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cbcb0-0eb9-7173-9a2e-3d7c7a1aacdd-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 27, 'output_tokens': 22, 'total_tokens': 49, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [97]:
print(chain_strategy.invoke({'tools':'''
1. Python
2. R Programming
3. SQL
4. Tableau
5. Hadoop
'''}))

1. Python: Start with the basics of Python, such as variables, data types, operators, and control flow. Once you have a good understanding of these, move on to more complex topics like functions, classes, and error handling. Use online resources like Codecademy, Coursera, or Python's official documentation to learn. Practice coding regularly, work on small projects, and try to solve problems on websites like HackerRank or LeetCode.

2. R Programming: Start with the basics


In [59]:
chain_combined = chain_tools | chain_strategy

In [60]:
print(chain_combined.invoke({'job title':'data scientist'}))

1. Python: Start with the basics of Python, such as variables, data types, loops, and functions. Once you have a good understanding of these, move on to more complex topics like classes and objects. Use online resources like Codecademy, Coursera, or Python's official documentation. Practice coding regularly and work on small projects to apply what you've learned. 

2. R Programming: Similar to Python, start with the basics of R. Learn about data structures, loops, and functions. Use online resources like DataCamp, Coursera, or R's official documentation. Practice by analyzing datasets and creating visualizations. 

3. SQL: Start by learning the basics of SQL, such as creating tables, inserting data, and querying data. Use online resources like Khan Academy, Codecademy, or SQL's official documentation. Practice by creating your own databases and running queries on them. 

4. Tableau: Start by learning the basics of Tableau, such as creating charts, maps, and dashboards. Use online resou

In [61]:
chain_long = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()} | 
              chat_template_strategy | chat | string_parser)

In [63]:
# Invoking the chain with a single input
result = chain_long.invoke({'job title': 'data scientist'})
print(result)

1. Python: Start with the basics of Python, such as variables, data types, operators, control flow, and functions. Once you have a good understanding of these, move on to more advanced topics like classes, exceptions, and modules. Use online resources like Codecademy, Coursera, or edX for structured learning. Practice coding regularly on platforms like LeetCode or HackerRank. Work on small projects to apply what you've learned.

2. R Programming: Start with the basics of R, such as data types, variables, vectors, and matrices. Then, move on to more advanced topics like data frames, lists, and factors. Use online resources like DataCamp, Coursera, or edX for structured learning. Practice coding regularly on platforms like R-exercises or Kaggle. Work on data analysis or data visualization projects to apply what you've learned.

3. SQL: Start with the basics of SQL, such as SELECT, FROM, WHERE, GROUP BY, and ORDER BY clauses. Then, move on to more advanced topics like joins, subqueries, a

In [64]:
chain_long.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *       

## RunnableParallel

In [65]:
from langchain_core.runnables import RunnableParallel

In [66]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books. 
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
    Answer only by listing the projects.
    '''
)

In [67]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [68]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [69]:
chain_parallel.invoke({'programming language':'Python'})

{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Web Scraper using BeautifulSoup and Requests.\n2. Developing a simple Machine Learning application using Scikit-learn.\n3. Creating a GUI application with Tkinter.'}

In [70]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [71]:
%%time
chain_books.invoke({'programming language':'Python'})

CPU times: user 14.7 ms, sys: 5.14 ms, total: 19.8 ms
Wall time: 2.42 s


'1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin'

In [72]:
%%time
chain_projects.invoke({'programming language':'Python'})

CPU times: user 13.8 ms, sys: 3.55 ms, total: 17.3 ms
Wall time: 1.88 s


'1. Building a Web Scraper using BeautifulSoup and Requests\n2. Developing a Text-Based Adventure Game using Object-Oriented Programming\n3. Creating a Personal Finance Tracker with Data Visualization using Matplotlib and Pandas'

In [73]:
%%time
chain_parallel.invoke({'programming language':'Python'})

CPU times: user 13.4 ms, sys: 21 ms, total: 34.4 ms
Wall time: 3.4 s


{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Web Scraper using BeautifulSoup and Requests\n2. Developing a simple Machine Learning application with Scikit-learn\n3. Creating a GUI application with Tkinter or PyQt'}

In [75]:
chat_template_time = ChatPromptTemplate.from_template(
     '''
     I'm an intermediate level programmer.
     
     Consider the following literature:
     {books}
     
     Also, consider the following projects:
     {projects}
     
     Roughly how much time would it take me to complete the literature and the projects?
     
     '''
)

In [76]:
chain_time1 = (RunnableParallel({'books':chain_books, 
                                'projects':chain_projects}) 
              | chat_template_time 
              | chat 
              | string_parser
             )

In [77]:
chain_time2 = ({'books':chain_books, 
                'projects':chain_projects}
              | chat_template_time 
              | chat 
              | string_parser
             )

In [106]:
print(chain_time2.invoke({'programming language':'Python'}))

The time it takes to complete the literature and the projects can vary greatly depending on several factors such as your current skill level, the amount of time you can dedicate each day, your reading speed, and how quickly you grasp new concepts. 

However, as a rough estimate:

1. "Fluent Python: Clear, Concise, and Effective Programming" - This book is around 800 pages. If you read and practice for about 2 hours a day, it might take you around 1


In [108]:
print(chain_time1.invoke({'programming language':'Python'}))

The time it would take to complete the literature and the projects can vary greatly depending on several factors such as your reading speed, comprehension level, the complexity of the projects, and the amount of time you can dedicate each day. 

For the literature, an average reader can read about 200-250 words per minute. Considering that each of these books is roughly 500-600 pages, it would take approximately 20-30 hours to read each book. So, for all three books, you


In [107]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [109]:
chain_time1.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

## RunnableLambda

In [79]:
from langchain_core.runnables import RunnableLambda

In [80]:
find_sum = lambda x: sum(x)

In [81]:
find_sum([1, 2, 5])

8

In [82]:
find_square = lambda x: x**2

In [83]:
find_square(8)

64

In [84]:
runnable_sum = RunnableLambda(lambda x: sum(x))

In [85]:
runnable_sum.invoke([1, 2, 5])

8

In [86]:
runnable_square = RunnableLambda(lambda x: x**2)

In [87]:
runnable_square.invoke(8)

64

In [90]:
chain = runnable_sum | runnable_square

In [91]:
chain.invoke([1, 2, 5])

64

In [121]:
chain.get_graph().print_ascii()

+-------------+  
| LambdaInput |  
+-------------+  
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
+--------------+ 
| LambdaOutput | 
+--------------+ 


## The @chain Decorator

In [122]:
# Run the line of code below to check the version of langchain in the current environment.
# Substitute "langchain" with any other package name to check their version.

In [92]:
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables import chain

In [ ]:
def find_sum(x):
    return sum(x)

def find_square(x):
    return x**2

In [93]:
chain1 = RunnableLambda(find_sum) | RunnableLambda(find_square)

In [94]:
chain1.invoke([1, 2, 5])

64

In [99]:
@chain
def runnable_sum(x):
    return sum(x)

@chain
def runnable_square(x):
    return x**2

In [100]:
type(runnable_sum), type(runnable_square)

(langchain_core.runnables.base.RunnableLambda,
 langchain_core.runnables.base.RunnableLambda)

In [101]:
chain2 = runnable_sum | runnable_square

In [102]:
chain2.invoke([1, 2, 5])

64